# 00 — Data Overview

Comprehensive survey of the `data-gold-final/` data repository: directory structure, participant
coverage, feature extraction completeness, annotation inventory, clinical data, and experiment
outputs. This notebook accompanies `DATA_INVENTORY.md` with interactive visualizations.

**Sections:**
1. **Directory Structure** — Top-level layout and disk usage
2. **Participant Inventory** — Folder naming, cohort composition, modality coverage
3. **Feature Extraction Coverage** — Which features exist for which participants
4. **Annotations** — BORIS behavioral coding and pigeon prototype annotations
5. **Clinical Data** — Cohort demographics, diagnostic groups, assessment scores
6. **Experiment Outputs** — Recursive prototyping rounds, CPD experiments, clustering deployments
7. **Quality Control** — Visual inspection results and data corrections
8. **Data Completeness Matrix** — Summary heatmap of all data availability

In [ ]:
import json
import os
import re
from collections import Counter, defaultdict
from glob import glob
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

from smartflat.utils.utils_io import get_data_root

sns.set_theme(style="whitegrid", palette="muted")
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 80)

In [ ]:
# Data root — resolved via hostname or SMARTFLAT_DATA_ROOT env var
data_root = get_data_root()
print(f"Data root: {data_root}")
assert os.path.isdir(data_root), f"Data root not found: {data_root}"

## 1. Directory Structure

Top-level layout of `data-gold-final/` and disk usage per directory.

In [ ]:
# Top-level directory listing with entry counts
top_dirs = sorted([
    d for d in os.listdir(data_root)
    if os.path.isdir(os.path.join(data_root, d))
])

rows = []
for d in top_dirs:
    dpath = os.path.join(data_root, d)
    entries = os.listdir(dpath)
    n_files = sum(1 for e in entries if os.path.isfile(os.path.join(dpath, e)))
    n_dirs = sum(1 for e in entries if os.path.isdir(os.path.join(dpath, e)))
    rows.append({"directory": d, "subdirectories": n_dirs, "files": n_files})

df_top = pd.DataFrame(rows)
print(f"Top-level directories in data-gold-final/ ({len(top_dirs)} total):\n")
display(df_top)

In [ ]:
# Disk usage per top-level directory (may take a moment for large dirs)
import subprocess

disk_rows = []
for d in top_dirs:
    dpath = os.path.join(data_root, d)
    try:
        result = subprocess.run(
            ["du", "-sh", dpath], capture_output=True, text=True, timeout=60
        )
        size_str = result.stdout.strip().split("\t")[0] if result.stdout else "?"
    except Exception:
        size_str = "?"
    disk_rows.append({"directory": d, "size": size_str})

df_disk = pd.DataFrame(disk_rows)
print("Disk usage per top-level directory:\n")
display(df_disk)

## 2. Participant Inventory

Parse participant folder names in `cuisine/` to extract group IDs, participant types
(C=control, P=patient), trigrams, and recording dates. Visualize cohort composition.

In [ ]:
# Parse participant folder names: G{id}_{C|P}{pid}_{trigram}_{date}
cuisine_dir = os.path.join(data_root, "cuisine")
participant_folders = sorted([
    d for d in os.listdir(cuisine_dir)
    if os.path.isdir(os.path.join(cuisine_dir, d)) and d.startswith("G")
])

FOLDER_PATTERN = re.compile(
    r"^G(\d+)_([CP])(\d+)_([A-Z]{3}[A-Za-z]{3})_(\d{8})$"
)

rows = []
unparsed = []
for folder in participant_folders:
    m = FOLDER_PATTERN.match(folder)
    if m:
        group_id, ptype, pid, trigram, date_str = m.groups()
        try:
            rec_date = pd.to_datetime(date_str, format="%d%m%Y")
        except ValueError:
            rec_date = pd.NaT
        rows.append({
            "folder": folder,
            "group_id": int(group_id),
            "participant_type": "Control" if ptype == "C" else "Patient",
            "participant_id": int(pid),
            "trigram": trigram,
            "recording_date": rec_date,
        })
    else:
        unparsed.append(folder)

df_participants = pd.DataFrame(rows)
print(f"Participant folders: {len(participant_folders)}")
print(f"  Successfully parsed: {len(df_participants)}")
print(f"  Unparsed: {len(unparsed)}")
if unparsed:
    print(f"  Unparsed folders: {unparsed[:10]}")
display(df_participants.head(10))

In [ ]:
# Cohort composition
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Controls vs Patients
counts = df_participants["participant_type"].value_counts()
axes[0].bar(counts.index, counts.values, color=["#4C72B0", "#DD8452"], edgecolor="black")
for i, (label, val) in enumerate(zip(counts.index, counts.values)):
    axes[0].text(i, val + 1, str(val), ha="center", fontweight="bold")
axes[0].set(title="Cohort Composition", ylabel="Count")

# Recordings over time (by month)
if df_participants["recording_date"].notna().any():
    df_participants["year_month"] = df_participants["recording_date"].dt.to_period("M")
    monthly = df_participants.groupby(["year_month", "participant_type"]).size().unstack(fill_value=0)
    monthly.index = monthly.index.astype(str)
    monthly.plot.bar(ax=axes[1], stacked=True, edgecolor="black", width=0.8)
    axes[1].set(title="Recordings Over Time", ylabel="Count", xlabel="")
    axes[1].tick_params(axis="x", rotation=90, labelsize=7)
    axes[1].legend(title="", fontsize=8)

# Group ID distribution
axes[2].hist(df_participants["group_id"], bins=30, edgecolor="black", alpha=0.7)
axes[2].set(title="Group ID Distribution", xlabel="Group ID (G-number)", ylabel="Count")

plt.tight_layout()
plt.show()

print(f"\nSummary:")
print(f"  Controls: {(df_participants['participant_type'] == 'Control').sum()}")
print(f"  Patients: {(df_participants['participant_type'] == 'Patient').sum()}")
if df_participants['recording_date'].notna().any():
    print(f"  Date range: {df_participants['recording_date'].min().date()} → "
          f"{df_participants['recording_date'].max().date()}")

In [ ]:
# Modality subfolder coverage per participant
EXPECTED_MODALITIES = ["GoPro1", "GoPro2", "GoPro3", "Tobii", "Annotation", "Audacity"]

modality_rows = []
for folder in participant_folders:
    fpath = os.path.join(cuisine_dir, folder)
    present = set(os.listdir(fpath))
    row = {"folder": folder}
    for mod in EXPECTED_MODALITIES:
        row[mod] = mod in present
    modality_rows.append(row)

df_modality = pd.DataFrame(modality_rows)

# Summary: how many participants have each modality folder
mod_counts = df_modality[EXPECTED_MODALITIES].sum()
print("Modality folder presence across participants:\n")
display(mod_counts.to_frame("n_participants").T)

# Heatmap of modality presence (sorted by group ID for readability)
fig, ax = plt.subplots(figsize=(8, max(4, len(df_modality) * 0.06)))
sns.heatmap(
    df_modality[EXPECTED_MODALITIES].astype(int).values,
    yticklabels=[],
    xticklabels=EXPECTED_MODALITIES,
    cmap="YlGn",
    cbar_kws={"label": "Present", "ticks": [0, 1]},
    ax=ax,
)
ax.set(title=f"Modality Folder Presence ({len(df_modality)} participants)", ylabel="Participant")
plt.tight_layout()
plt.show()

## 3. Feature Extraction Coverage

Scan Tobii folders (primary modality) for extracted feature files and flag files.
Each feature type has a corresponding flag file (`.{video}_{type}_flag.txt`) set on completion.

See: `smartflat.utils.utils_paths.fetch_output_path`, `smartflat.utils.utils_paths.fetch_flag_path`

In [ ]:
# Scan Tobii folders for feature files and flag files
FEATURE_PATTERNS = {
    "video_representation": "video_representations_VideoMAEv2_*.npy",
    "hand_landmarks": "hand_landmarks_mediapipe_*.json",
    "skeleton_landmarks": "skeleton_landmarks_*.json",
    "speech_recognition": "speech_recognition_diarization_whisperx_*.json",
    "speech_representation": "speech_representations_multilingual_*.npy",
    "tracking_hand_landmarks": "tracking_hand_landmarks_*.json",
}

FLAG_PATTERN = "._flag.txt"  # hidden flag files end with _flag.txt

feature_rows = []
for folder in participant_folders:
    tobii_dir = os.path.join(cuisine_dir, folder, "Tobii")
    row = {"folder": folder}
    if os.path.isdir(tobii_dir):
        all_files = os.listdir(tobii_dir)
        for feat_name, pattern in FEATURE_PATTERNS.items():
            # Check for feature files matching the glob pattern
            prefix = pattern.split("*")[0]
            suffix = pattern.split("*")[1]
            has_feature = any(f.startswith(prefix) and f.endswith(suffix) for f in all_files)
            # Check for flag files
            has_flag = any(f.endswith(f"_{feat_name}_flag.txt") for f in all_files)
            row[feat_name] = has_feature
            row[f"{feat_name}_flag"] = has_flag
        # Count raw video files
        row["has_video"] = any(f.endswith(".mp4") for f in all_files)
        row["has_audio"] = any(f.endswith(".wav") for f in all_files)
    else:
        for feat_name in FEATURE_PATTERNS:
            row[feat_name] = False
            row[f"{feat_name}_flag"] = False
        row["has_video"] = False
        row["has_audio"] = False
    feature_rows.append(row)

df_features = pd.DataFrame(feature_rows)
print(f"Feature extraction scan: {len(df_features)} participants\n")

In [ ]:
# Feature coverage summary
feature_cols = list(FEATURE_PATTERNS.keys()) + ["has_video", "has_audio"]
coverage = df_features[feature_cols].sum()

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(
    range(len(coverage)),
    coverage.values,
    color=sns.color_palette("muted", len(coverage)),
    edgecolor="black",
)
ax.set_xticks(range(len(coverage)))
ax.set_xticklabels(coverage.index, rotation=35, ha="right")
ax.axhline(len(df_features), color="red", ls="--", alpha=0.5,
           label=f"Total participants ({len(df_features)})")
for i, val in enumerate(coverage.values):
    ax.text(i, val + 1, str(val), ha="center", fontsize=9)
ax.set(title="Feature Extraction Coverage (Tobii modality)", ylabel="Participants with feature")
ax.legend()
plt.tight_layout()
plt.show()

# Table view
pct = (coverage / len(df_features) * 100).round(1)
display(pd.DataFrame({"count": coverage, "pct": pct}).T)

In [ ]:
# Per-participant feature completeness heatmap
feat_matrix = df_features[feature_cols].astype(int)

fig, ax = plt.subplots(figsize=(10, max(4, len(feat_matrix) * 0.06)))
sns.heatmap(
    feat_matrix.values,
    yticklabels=[],
    xticklabels=feature_cols,
    cmap="YlGn",
    cbar_kws={"label": "Present", "ticks": [0, 1]},
    ax=ax,
)
ax.set(title=f"Feature Extraction Completeness ({len(feat_matrix)} participants)", ylabel="Participant")
plt.tight_layout()
plt.show()

# Participants with ALL core features (video_representation + hand_landmarks)
core_feats = ["video_representation", "hand_landmarks"]
n_complete = df_features[core_feats].all(axis=1).sum()
print(f"\nParticipants with both video_representation + hand_landmarks: "
      f"{n_complete}/{len(df_features)} ({n_complete/len(df_features)*100:.1f}%)")

# Participants missing video representation
missing_vr = df_features[~df_features["video_representation"]]["folder"].tolist()
if missing_vr:
    print(f"\nMissing video_representation ({len(missing_vr)}):")
    for f in missing_vr[:10]:
        print(f"  {f}")
    if len(missing_vr) > 10:
        print(f"  ... and {len(missing_vr) - 10} more")

In [ ]:
# Feature file sizes (sample a few participants for size distribution)
size_rows = []
for folder in participant_folders[:50]:  # sample first 50 for speed
    tobii_dir = os.path.join(cuisine_dir, folder, "Tobii")
    if not os.path.isdir(tobii_dir):
        continue
    for f in os.listdir(tobii_dir):
        fpath = os.path.join(tobii_dir, f)
        if not os.path.isfile(fpath) or f.startswith("."):
            continue
        ext = os.path.splitext(f)[1]
        # Classify by feature type
        if "video_representations" in f:
            ftype = "video_repr (.npy)"
        elif "hand_landmarks_mediapipe" in f:
            ftype = "hand_landmarks (.json)"
        elif "skeleton_landmarks" in f:
            ftype = "skeleton (.json)"
        elif "speech_recognition" in f:
            ftype = "speech_recog (.json)"
        elif "speech_representations" in f:
            ftype = "speech_repr (.npy)"
        elif ext == ".mp4":
            ftype = "raw_video (.mp4)"
        elif ext == ".wav":
            ftype = "raw_audio (.wav)"
        else:
            continue
        size_mb = os.path.getsize(fpath) / (1024 ** 2)
        size_rows.append({"folder": folder, "type": ftype, "size_mb": size_mb})

if size_rows:
    df_sizes = pd.DataFrame(size_rows)
    fig, ax = plt.subplots(figsize=(10, 5))
    order = df_sizes.groupby("type")["size_mb"].median().sort_values(ascending=False).index
    sns.boxplot(data=df_sizes, x="type", y="size_mb", order=order, ax=ax)
    ax.set(title="File Size Distribution by Feature Type (sample of 50 participants)",
           xlabel="", ylabel="Size (MB)")
    ax.tick_params(axis="x", rotation=30)
    plt.tight_layout()
    plt.show()

    print("\nMedian file sizes:")
    display(df_sizes.groupby("type")["size_mb"].agg(["median", "mean", "min", "max"]).round(1))

## 4. Annotations

### 4.1 BORIS Behavioral Annotations

Consolidated BORIS annotations covering the A–J activity categories for the cooking task.

See: `smartflat.annotation_smartflat`

In [ ]:
# Load BORIS annotations
annotations_path = os.path.join(data_root, "dataframes", "annotations", "annotations_all.csv")
if os.path.isfile(annotations_path):
    df_annot = pd.read_csv(annotations_path)
    print(f"BORIS annotations: {len(df_annot)} rows, {df_annot.shape[1]} columns")
    print(f"Columns: {list(df_annot.columns)}\n")
    display(df_annot.head(5))
else:
    df_annot = None
    print(f"Annotations file not found at {annotations_path}")

In [ ]:
# Annotation statistics
if df_annot is not None:
    # Identify the behavior/label column (varies across annotation exports)
    label_candidates = [c for c in df_annot.columns if "behav" in c.lower() or "label" in c.lower()
                        or "category" in c.lower() or "code" in c.lower()]
    id_candidates = [c for c in df_annot.columns if "subject" in c.lower() or "participant" in c.lower()
                     or "observation" in c.lower() or "identifier" in c.lower()]

    print(f"Candidate label columns: {label_candidates}")
    print(f"Candidate ID columns: {id_candidates}")

    if label_candidates:
        label_col = label_candidates[0]
        print(f"\nUsing label column: '{label_col}'")
        vc = df_annot[label_col].value_counts()
        print(f"Unique labels: {len(vc)}\n")

        fig, ax = plt.subplots(figsize=(12, 5))
        vc.head(20).plot.barh(ax=ax, edgecolor="black")
        ax.set(title=f"Top 20 Annotation Labels ('{label_col}')", xlabel="Count")
        ax.invert_yaxis()
        plt.tight_layout()
        plt.show()

    if id_candidates:
        id_col = id_candidates[0]
        n_annotated = df_annot[id_col].nunique()
        print(f"\nAnnotated subjects ('{id_col}'): {n_annotated}")

### 4.2 Pigeon Prototype Annotations

Manual annotations from the recursive prototyping procedure (Ch. 5). Annotators qualified
each cluster prototype as: `task-definitive`, `exo-definitive`, `task-ambiguous`,
`exo-ambiguous`, or `Noise` across 8 rounds.

See: `dataframes/annotations/pigeon-annotations/symbolization-gold/`

In [ ]:
# Inventory pigeon annotations: which experiment_ids, annotators, and rounds exist
pigeon_root = os.path.join(data_root, "dataframes", "annotations", "pigeon-annotations")
pigeon_gold = os.path.join(pigeon_root, "symbolization-gold")

pigeon_rows = []
if os.path.isdir(pigeon_gold):
    for exp_id in sorted(os.listdir(pigeon_gold)):
        exp_path = os.path.join(pigeon_gold, exp_id)
        if not os.path.isdir(exp_path):
            continue
        for annotator in sorted(os.listdir(exp_path)):
            ann_path = os.path.join(exp_path, annotator)
            if not os.path.isdir(ann_path):
                # CSV files directly under experiment_id/
                if annotator.endswith(".csv"):
                    pigeon_rows.append({
                        "experiment_id": exp_id,
                        "annotator": "—",
                        "round": annotator,
                        "file": annotator,
                    })
                continue
            for f in sorted(os.listdir(ann_path)):
                if f.endswith(".csv"):
                    pigeon_rows.append({
                        "experiment_id": exp_id,
                        "annotator": annotator,
                        "round": f,
                        "file": f,
                    })

df_pigeon = pd.DataFrame(pigeon_rows)
if len(df_pigeon) > 0:
    print(f"Pigeon annotation files: {len(df_pigeon)}")
    print(f"Experiment IDs: {df_pigeon['experiment_id'].nunique()}")
    print(f"Annotators: {df_pigeon['annotator'].unique().tolist()}\n")
    display(df_pigeon.groupby(["experiment_id", "annotator"]).size()
            .reset_index(name="n_rounds"))
else:
    print("No pigeon annotations found.")

In [ ]:
# Load and visualize annotation category distribution across rounds
# Focus on the main production experiment: faissc_refinement_symbolization
main_exp = "faissc_refinement_symbolization"
main_annotator = "samperochon"
ann_dir = os.path.join(pigeon_gold, main_exp, main_annotator)

round_dfs = {}
if os.path.isdir(ann_dir):
    for f in sorted(os.listdir(ann_dir)):
        if not f.endswith(".csv"):
            continue
        # Extract round number from filename: round_N_prototypes_K_100.csv
        m = re.search(r"round_(\d+)", f)
        if m:
            rnum = int(m.group(1))
            try:
                rdf = pd.read_csv(os.path.join(ann_dir, f), sep=";")
                round_dfs[rnum] = rdf
            except Exception as e:
                print(f"  Failed to load {f}: {e}")

if round_dfs:
    print(f"Loaded {len(round_dfs)} annotation rounds for {main_exp}/{main_annotator}\n")

    # Find the cluster_type column
    sample = list(round_dfs.values())[0]
    type_col = [c for c in sample.columns if "cluster_type" in c.lower() or "type" in c.lower()
                or "annotation" in c.lower() or "label" in c.lower()]
    if type_col:
        type_col = type_col[0]
        print(f"Using annotation column: '{type_col}'")
        print(f"Categories: {sample[type_col].unique().tolist()}\n")

        # Build category counts per round
        cat_data = {}
        for rnum, rdf in sorted(round_dfs.items()):
            cat_data[f"Round {rnum}"] = rdf[type_col].value_counts()

        df_cats = pd.DataFrame(cat_data).fillna(0).astype(int)

        fig, axes = plt.subplots(1, 2, figsize=(16, 5))

        # Stacked bar: categories per round
        df_cats.T.plot.bar(stacked=True, ax=axes[0], edgecolor="black", width=0.8)
        axes[0].set(title=f"Annotation Categories per Round\n({main_exp})",
                    ylabel="Count", xlabel="")
        axes[0].legend(title="Category", fontsize=7, title_fontsize=8, loc="upper left")
        axes[0].tick_params(axis="x", rotation=45)

        # Line: cumulative prototypes by category
        cum_cats = df_cats.T.cumsum()
        cum_cats.plot(ax=axes[1], marker="o", linewidth=2)
        axes[1].set(title="Cumulative Prototypes by Category",
                    ylabel="Cumulative count", xlabel="")
        axes[1].legend(title="Category", fontsize=7, title_fontsize=8)
        axes[1].tick_params(axis="x", rotation=45)

        plt.tight_layout()
        plt.show()

        print("Category counts per round:")
        display(df_cats)
    else:
        print(f"No type column found. Columns: {sample.columns.tolist()}")
else:
    print(f"No annotation rounds found at {ann_dir}")

## 5. Clinical Data

Load and summarize the clinical assessment data: diagnostic groups (Control/TBI/RIL),
demographics, and neuropsychological test scores.

See: `dataframes/clinical/`

In [ ]:
# Load clinical metadata
clinical_dir = os.path.join(data_root, "dataframes", "clinical")
clinical_files = sorted(os.listdir(clinical_dir)) if os.path.isdir(clinical_dir) else []
print(f"Clinical data files ({len(clinical_files)}):")
for f in clinical_files:
    if os.path.isfile(os.path.join(clinical_dir, f)):
        size_kb = os.path.getsize(os.path.join(clinical_dir, f)) / 1024
        print(f"  {f:55s} {size_kb:8.1f} KB")

In [ ]:
# Load clinical_metadata.csv — simplified group assignments
clinical_meta_path = os.path.join(clinical_dir, "clinical_metadata.csv")
if os.path.isfile(clinical_meta_path):
    df_clinical = pd.read_csv(clinical_meta_path)
    print(f"Clinical metadata: {len(df_clinical)} rows, {df_clinical.shape[1]} columns")
    print(f"Columns: {list(df_clinical.columns)}\n")
    display(df_clinical.head(10))
else:
    df_clinical = None
    print(f"Not found: {clinical_meta_path}")

In [ ]:
# Diagnostic group distribution
if df_clinical is not None:
    # Find the group/pathologie column
    group_col = None
    for candidate in ["pathologie", "groupe", "group", "diagnosis"]:
        if candidate in df_clinical.columns:
            group_col = candidate
            break

    if group_col:
        group_counts = df_clinical[group_col].value_counts()

        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        # Bar chart
        colors = {"Control": "#4C72B0", "Patient": "#DD8452", "TBI": "#DD8452",
                  "RIL": "#55A868", "TC": "#DD8452"}
        bar_colors = [colors.get(g, "#999999") for g in group_counts.index]
        axes[0].bar(group_counts.index, group_counts.values, color=bar_colors, edgecolor="black")
        for i, (label, val) in enumerate(zip(group_counts.index, group_counts.values)):
            axes[0].text(i, val + 0.5, str(val), ha="center", fontweight="bold")
        axes[0].set(title=f"Diagnostic Groups ('{group_col}')", ylabel="Count")

        # Pie chart
        axes[1].pie(group_counts.values, labels=group_counts.index, autopct="%1.1f%%",
                    colors=bar_colors, startangle=90, textprops={"fontsize": 10})
        axes[1].set_title("Group Proportions")

        plt.tight_layout()
        plt.show()

    # Numeric score distributions (if available)
    numeric_cols = df_clinical.select_dtypes(include=[np.number]).columns.tolist()
    # Exclude ID-like columns
    score_cols = [c for c in numeric_cols if not any(
        kw in c.lower() for kw in ["id", "index", "unnamed"]
    )]

    if score_cols and group_col:
        n_scores = min(len(score_cols), 6)
        fig, axes = plt.subplots(2, 3, figsize=(15, 8))
        axes = axes.flatten()
        for i, col in enumerate(score_cols[:n_scores]):
            sns.boxplot(data=df_clinical, x=group_col, y=col, ax=axes[i])
            axes[i].set_title(col, fontsize=10)
        for i in range(n_scores, len(axes)):
            axes[i].set_visible(False)
        plt.suptitle("Clinical Score Distributions by Group", y=1.02, fontsize=13)
        plt.tight_layout()
        plt.show()

In [ ]:
# Load detailed clinical scores (MUPT)
mupt_path = os.path.join(clinical_dir, "clinical_data_mupt.csv")
if os.path.isfile(mupt_path):
    # Try multiple separators
    for sep in [",", ";", "\t"]:
        try:
            df_mupt = pd.read_csv(mupt_path, sep=sep)
            if df_mupt.shape[1] > 2:
                break
        except Exception:
            continue

    print(f"MUPT clinical data: {len(df_mupt)} rows, {df_mupt.shape[1]} columns")
    print(f"Columns: {list(df_mupt.columns)}\n")
    display(df_mupt.head(5))

    # Summary statistics for numeric scores
    numeric_mupt = df_mupt.select_dtypes(include=[np.number])
    if not numeric_mupt.empty:
        print(f"\nNumeric score summary ({numeric_mupt.shape[1]} columns):")
        display(numeric_mupt.describe().round(2))
else:
    print(f"MUPT data not found: {mupt_path}")

## 6. Experiment Outputs

Inventory of experiment result directories: recursive prototyping (symbolization-gold),
change-point detection, and clustering deployments. Each experiment follows the path
pattern `{experiment_name}/{experiment_id}/{annotator_id}/round_{N}/`.

See: `configs/symbolic_config_gold.py`, `configs/change_points_config.py`, `configs/clustering_config.py`

In [ ]:
# Inventory experiments/ directory
experiments_dir = os.path.join(data_root, "experiments")

exp_rows = []
for name in sorted(os.listdir(experiments_dir)):
    epath = os.path.join(experiments_dir, name)
    if not os.path.isdir(epath):
        continue
    # Count subdirectories at next level
    subs = [s for s in os.listdir(epath) if os.path.isdir(os.path.join(epath, s))]
    files = [f for f in os.listdir(epath) if os.path.isfile(os.path.join(epath, f))]
    # Classify experiment type
    if "change-point" in name:
        exp_type = "Change Point Detection"
    elif "clustering" in name:
        exp_type = "Clustering"
    elif "symbolization" in name:
        exp_type = "Symbolization"
    elif name == "experiments":
        exp_type = "LEGACY (nested)"
    else:
        exp_type = "Other"
    exp_rows.append({
        "experiment_name": name,
        "type": exp_type,
        "n_subdirs": len(subs),
        "n_files": len(files),
    })

df_experiments = pd.DataFrame(exp_rows)
print(f"Experiment directories: {len(df_experiments)}\n")
display(df_experiments)

# Visualize by type
fig, ax = plt.subplots(figsize=(10, 4))
type_counts = df_experiments["type"].value_counts()
type_counts.plot.barh(ax=ax, edgecolor="black", color=sns.color_palette("muted"))
ax.set(title="Experiment Directories by Type", xlabel="Count")
plt.tight_layout()
plt.show()

In [ ]:
# Deep-dive: symbolization-gold experiment structure
symb_exp_dir = os.path.join(experiments_dir, "symbolization-gold")
symb_exp_ids = sorted([
    d for d in os.listdir(symb_exp_dir)
    if os.path.isdir(os.path.join(symb_exp_dir, d))
]) if os.path.isdir(symb_exp_dir) else []

# For each experiment_id, check which annotators and rounds exist
symb_rows = []
for exp_id in symb_exp_ids:
    eid_path = os.path.join(symb_exp_dir, exp_id)
    annotators = [d for d in os.listdir(eid_path)
                  if os.path.isdir(os.path.join(eid_path, d))]
    for ann in annotators:
        ann_path = os.path.join(eid_path, ann)
        rounds = sorted([
            d for d in os.listdir(ann_path)
            if os.path.isdir(os.path.join(ann_path, d)) and d.startswith("round_")
        ])
        round_nums = [int(r.split("_")[1]) for r in rounds]
        # Count files in latest round
        if rounds:
            latest = os.path.join(ann_path, rounds[-1])
            n_files = len([f for f in os.listdir(latest) if os.path.isfile(os.path.join(latest, f))])
        else:
            n_files = 0
        symb_rows.append({
            "experiment_id": exp_id,
            "annotator": ann,
            "rounds": round_nums,
            "n_rounds": len(rounds),
            "files_in_latest": n_files,
        })

df_symb = pd.DataFrame(symb_rows)
print(f"Symbolization-gold experiments: {len(symb_exp_ids)} experiment_ids\n")
display(df_symb)

In [ ]:
# Visualize round coverage across experiment_ids (for samperochon)
sam_symb = df_symb[df_symb["annotator"] == "samperochon"].copy()
if len(sam_symb) > 0:
    # Create a round coverage matrix
    all_rounds = sorted(set(r for rounds in sam_symb["rounds"] for r in rounds))
    round_matrix = []
    exp_labels = []
    for _, row in sam_symb.iterrows():
        round_matrix.append([1 if r in row["rounds"] else 0 for r in all_rounds])
        exp_labels.append(row["experiment_id"])

    round_arr = np.array(round_matrix)

    fig, ax = plt.subplots(figsize=(10, max(4, len(exp_labels) * 0.35)))
    sns.heatmap(
        round_arr,
        yticklabels=exp_labels,
        xticklabels=[f"R{r}" for r in all_rounds],
        cmap="YlGn",
        cbar_kws={"label": "Present", "ticks": [0, 1]},
        ax=ax,
        linewidths=0.5,
    )
    ax.set(title="Symbolization-Gold Round Coverage (samperochon)")
    plt.tight_layout()
    plt.show()

In [ ]:
# Change-point detection experiments: count UUID folders (grid search runs)
cpd_experiments = {
    "change-point-detection-experiment": "Raw-space CPD grid search",
    "gold-change-point-detection-prototypes-experiment": "Prototype-space CPD grid search",
}

for folder, desc in cpd_experiments.items():
    cpd_path = os.path.join(experiments_dir, folder)
    if os.path.isdir(cpd_path):
        uuids = [d for d in os.listdir(cpd_path) if os.path.isdir(os.path.join(cpd_path, d))]
        # Sample a config.json from one UUID
        sample_config = None
        for uid in uuids[:5]:
            cfg_path = os.path.join(cpd_path, uid, "config.json")
            if os.path.isfile(cfg_path):
                with open(cfg_path) as f:
                    sample_config = json.load(f)
                break
        print(f"{folder}/")
        print(f"  Description: {desc}")
        print(f"  UUID folders: {len(uuids)}")
        if sample_config:
            print(f"  Sample config keys: {list(sample_config.keys())}")
        print()

In [ ]:
# Outputs directory inventory
outputs_dir = os.path.join(data_root, "outputs")
output_rows = []
for name in sorted(os.listdir(outputs_dir)):
    opath = os.path.join(outputs_dir, name)
    if not os.path.isdir(opath):
        continue
    subs = [s for s in os.listdir(opath) if os.path.isdir(os.path.join(opath, s))]
    files = [f for f in os.listdir(opath) if os.path.isfile(os.path.join(opath, f))]
    output_rows.append({
        "output_name": name,
        "n_subdirs": len(subs),
        "n_files": len(files),
    })

df_outputs = pd.DataFrame(output_rows)
print(f"Output directories: {len(df_outputs)}\n")
display(df_outputs)

In [ ]:
# Symbolization-gold outputs: list the state/config PKL files at root level
symb_out_dir = os.path.join(outputs_dir, "symbolization-gold")
if os.path.isdir(symb_out_dir):
    pkl_files = sorted([f for f in os.listdir(symb_out_dir)
                        if os.path.isfile(os.path.join(symb_out_dir, f)) and f.endswith(".pkl")])
    subdirs = sorted([d for d in os.listdir(symb_out_dir)
                      if os.path.isdir(os.path.join(symb_out_dir, d))])

    print(f"outputs/symbolization-gold/")
    print(f"  Subdirectories: {len(subdirs)}")
    print(f"  PKL state files: {len(pkl_files)}")
    print(f"\nState files:")
    for f in pkl_files:
        size_mb = os.path.getsize(os.path.join(symb_out_dir, f)) / (1024 ** 2)
        print(f"  {f:70s} {size_mb:8.1f} MB")
    print(f"\nSubdirectories (experiment_ids):")
    for d in subdirs:
        print(f"  {d}")

## 7. Quality Control

Visual inspection results tracking data quality: modality swaps, failed recordings,
and flag corrections.

See: `smartflat.datasets.quality_control`

In [ ]:
# Load the latest QC results
qc_dir = os.path.join(data_root, "dataframes", "quality-control")
qc_files = sorted(os.listdir(qc_dir)) if os.path.isdir(qc_dir) else []
print(f"Quality control files: {qc_files}\n")

# Use the latest file
qc_latest = os.path.join(qc_dir, "results_visual_inspection_2111124_last.csv")
if not os.path.isfile(qc_latest) and qc_files:
    qc_latest = os.path.join(qc_dir, qc_files[-1])

if os.path.isfile(qc_latest):
    df_qc = pd.read_csv(qc_latest)
    print(f"Latest QC file: {os.path.basename(qc_latest)}")
    print(f"  Rows: {len(df_qc)}, Columns: {df_qc.shape[1]}")
    print(f"  Columns: {list(df_qc.columns)}\n")
    display(df_qc.head(5))
else:
    df_qc = None
    print("No QC files found.")

In [ ]:
# QC issue summary
if df_qc is not None:
    # Look for swap/failed/correction columns
    bool_cols = [c for c in df_qc.columns if df_qc[c].dtype == bool
                 or set(df_qc[c].dropna().unique()).issubset({True, False, 0, 1, "True", "False"})]
    status_cols = [c for c in df_qc.columns if "swap" in c.lower() or "fail" in c.lower()
                   or "correct" in c.lower() or "flag" in c.lower() or "issue" in c.lower()]

    display_cols = list(set(bool_cols + status_cols))
    if display_cols:
        print(f"QC flag columns: {display_cols}\n")
        for col in display_cols:
            vc = df_qc[col].value_counts()
            print(f"  {col}: {dict(vc)}")

        # Visualize issue counts
        issue_counts = {}
        for col in display_cols:
            try:
                n_issues = df_qc[col].astype(bool).sum()
                if n_issues > 0:
                    issue_counts[col] = n_issues
            except (ValueError, TypeError):
                pass

        if issue_counts:
            fig, ax = plt.subplots(figsize=(10, 4))
            names = list(issue_counts.keys())
            values = list(issue_counts.values())
            ax.barh(names, values, edgecolor="black", color="#DD8452")
            for i, v in enumerate(values):
                ax.text(v + 0.5, i, str(v), va="center")
            ax.set(title="QC Issues Flagged", xlabel="Count")
            plt.tight_layout()
            plt.show()
    else:
        print("No boolean/flag columns found in QC data.")

## 8. Data Completeness Matrix

Combined view: for each participant, what raw data, features, annotations, and clinical
data are available. This is the key summary for understanding data readiness.

In [ ]:
# Build a unified completeness matrix
# Merge: participant info + modality presence + feature presence + clinical match

df_complete = df_participants[["folder", "participant_type", "trigram"]].copy()

# Add modality presence
for mod in EXPECTED_MODALITIES:
    df_complete[f"mod_{mod}"] = df_modality.set_index("folder")[mod].values

# Add feature presence
for feat in list(FEATURE_PATTERNS.keys()) + ["has_video", "has_audio"]:
    df_complete[f"feat_{feat}"] = df_features.set_index("folder")[feat].values

# Add clinical data match (by trigram)
if df_clinical is not None:
    trigram_col = [c for c in df_clinical.columns if "trigram" in c.lower()
                   or "trigramme" in c.lower()]
    if trigram_col:
        clinical_trigrams = set(df_clinical[trigram_col[0]].dropna().str.strip().values)
        df_complete["has_clinical"] = df_complete["trigram"].isin(clinical_trigrams)
    else:
        df_complete["has_clinical"] = False
else:
    df_complete["has_clinical"] = False

# Add BORIS annotation match
if df_annot is not None and id_candidates:
    id_col = id_candidates[0]
    annotated_ids = set(df_annot[id_col].dropna().unique())
    # Try matching by folder name or trigram
    df_complete["has_boris_annot"] = (
        df_complete["folder"].isin(annotated_ids) |
        df_complete["trigram"].isin(annotated_ids)
    )
else:
    df_complete["has_boris_annot"] = False

print(f"Completeness matrix: {len(df_complete)} participants, {df_complete.shape[1]} columns")
display(df_complete.head(5))

In [ ]:
# Completeness heatmap — all boolean columns
bool_cols = [c for c in df_complete.columns if df_complete[c].dtype == bool]
display_labels = [c.replace("mod_", "").replace("feat_", "").replace("has_", "") for c in bool_cols]

fig, ax = plt.subplots(figsize=(14, max(5, len(df_complete) * 0.06)))
sns.heatmap(
    df_complete[bool_cols].astype(int).values,
    yticklabels=[],
    xticklabels=display_labels,
    cmap="YlGn",
    cbar_kws={"label": "Available", "ticks": [0, 1]},
    ax=ax,
    linewidths=0,
)
ax.set(title=f"Data Completeness Matrix ({len(df_complete)} participants)",
       ylabel="Participant (sorted by group ID)")
plt.tight_layout()
plt.show()

In [ ]:
# Summary: completeness percentages by category
completeness_summary = df_complete[bool_cols].mean().round(3) * 100
completeness_summary.index = display_labels

fig, ax = plt.subplots(figsize=(12, 5))
colors = []
for label in display_labels:
    if label in EXPECTED_MODALITIES:
        colors.append("#4C72B0")  # modality
    elif label in ["clinical", "boris_annot"]:
        colors.append("#55A868")  # external data
    else:
        colors.append("#DD8452")  # features

bars = ax.bar(range(len(completeness_summary)), completeness_summary.values,
              color=colors, edgecolor="black")
ax.set_xticks(range(len(completeness_summary)))
ax.set_xticklabels(display_labels, rotation=40, ha="right")
ax.axhline(100, color="red", ls="--", alpha=0.3)
for i, val in enumerate(completeness_summary.values):
    ax.text(i, val + 1, f"{val:.0f}%", ha="center", fontsize=8)
ax.set(title="Data Completeness (% of participants)", ylabel="% Available",
       ylim=(0, 110))

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#4C72B0", edgecolor="black", label="Modality folders"),
    Patch(facecolor="#DD8452", edgecolor="black", label="Extracted features"),
    Patch(facecolor="#55A868", edgecolor="black", label="External data"),
]
ax.legend(handles=legend_elements, loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
# Completeness by participant type (Control vs Patient)
if "participant_type" in df_complete.columns:
    grouped = df_complete.groupby("participant_type")[bool_cols].mean().round(3) * 100
    grouped.columns = display_labels

    fig, ax = plt.subplots(figsize=(14, 5))
    x = np.arange(len(display_labels))
    width = 0.35
    ax.bar(x - width / 2, grouped.loc["Control"] if "Control" in grouped.index else [],
           width, label="Control", edgecolor="black", color="#4C72B0")
    ax.bar(x + width / 2, grouped.loc["Patient"] if "Patient" in grouped.index else [],
           width, label="Patient", edgecolor="black", color="#DD8452")
    ax.set_xticks(x)
    ax.set_xticklabels(display_labels, rotation=40, ha="right")
    ax.set(title="Data Completeness by Participant Type", ylabel="% Available", ylim=(0, 115))
    ax.axhline(100, color="red", ls="--", alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()

    print("\nCompleteness by group (%):")
    display(grouped)

## 9. Gold Dataset Metadata

Load the gold dataset metadata (the filtered, analysis-ready dataset) and compare
against the raw participant inventory.

See: `smartflat.datasets.loader.get_dataset`, `smartflat.datasets.filter`

In [ ]:
# Load the gold dataset backup CSV directly
gold_path = os.path.join(data_root, "dataframes", "gold_dataset_df_backup.csv")
if os.path.isfile(gold_path):
    df_gold = pd.read_csv(gold_path)
    print(f"Gold dataset: {len(df_gold)} rows, {df_gold.shape[1]} columns")
    print(f"Columns: {list(df_gold.columns)}\n")

    # Key statistics
    for col in ["participant_id", "task_name", "modality", "identifier"]:
        if col in df_gold.columns:
            print(f"  Unique {col}: {df_gold[col].nunique()}")

    display(df_gold.head(5))
else:
    df_gold = None
    print(f"Gold dataset not found at {gold_path}")

In [ ]:
# Gold dataset: recordings per task x modality, duration distributions
if df_gold is not None:
    print("Recordings per task x modality:")
    if "task_name" in df_gold.columns and "modality" in df_gold.columns:
        agg_cols = {"identifier": "nunique"}
        if "participant_id" in df_gold.columns:
            agg_cols["participant_id"] = "nunique"
        if "duration" in df_gold.columns:
            agg_cols["duration"] = ["mean", "std"]
        display(df_gold.groupby(["task_name", "modality"]).agg(agg_cols).round(2))

    # Duration distribution
    if "duration" in df_gold.columns and df_gold["duration"].notna().any():
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))

        # By task
        for task in sorted(df_gold["task_name"].unique()):
            durations = df_gold[df_gold["task_name"] == task]["duration"].dropna()
            axes[0].hist(durations, bins=30, alpha=0.6, edgecolor="black",
                         label=f"{task} (n={len(durations)})")
        axes[0].set(xlabel="Duration (min)", ylabel="Count",
                    title="Gold Dataset — Duration by Task")
        axes[0].legend()

        # By modality (cuisine only)
        cuisine = df_gold[df_gold["task_name"] == "cuisine"] if "task_name" in df_gold.columns else df_gold
        for mod in sorted(cuisine["modality"].unique()):
            durations = cuisine[cuisine["modality"] == mod]["duration"].dropna()
            axes[1].hist(durations, bins=30, alpha=0.6, edgecolor="black",
                         label=f"{mod} (n={len(durations)})")
        axes[1].set(xlabel="Duration (min)", ylabel="Count",
                    title="Cuisine — Duration by Modality")
        axes[1].legend()

        plt.tight_layout()
        plt.show()

    # Feature extraction flags in gold dataset
    flag_cols = [c for c in df_gold.columns if "flag" in c.lower() or "representation" in c.lower()]
    if flag_cols:
        print(f"\nFeature-related columns in gold dataset: {flag_cols}")

## 10. Legacy and Exploratory Data

Folders identified as legacy, exploratory, or misplaced — not part of the production pipeline.
See `DATA_INVENTORY.md` Appendix A for the full classification.

In [ ]:
# Inventory of legacy/exploratory folders
legacy_items = [
    ("cuisine/experiments/", "Misplaced experiment folder under cuisine/"),
    ("experiments/experiments/", "Nested duplicate of experiments/"),
    ("experiments/clustering-deployment-prototypes-v0/", "Superseded by v1"),
    ("outputs/symbolization-v1/", "Superseded by symbolization-gold"),
    ("outputs/symbolization-gold/zsocre_iteration_1", "Typo variant (zsocre → zscore)"),
    ("outputs/symbolization-gold/iteration_1", "Unnamed/unqualified iteration"),
    ("outputs/symbolization-gold/cuisine/", "Misplaced task-level subfolder"),
    ("experiments_speech/", "Not part of thesis pipeline"),
    ("thumbnails/thumbnails-cheetah/", "Non-cuisine task thumbnails"),
    ("thumbnails/thumbnails-pomme/", "Non-cuisine task thumbnails"),
    ("dataframes/backup_lego_cake_table_post_conversion.csv", "Lego task (non-cuisine)"),
    ("dataframes/lego_cake_table_with_id.csv", "Lego task (non-cuisine)"),
    ("dataframes/fix_DUMJEA.csv", "One-off data correction"),
]

legacy_rows = []
for path, reason in legacy_items:
    full_path = os.path.join(data_root, path)
    exists = os.path.exists(full_path)
    if exists and os.path.isdir(full_path):
        try:
            result = subprocess.run(
                ["du", "-sh", full_path], capture_output=True, text=True, timeout=30
            )
            size = result.stdout.strip().split("\t")[0] if result.stdout else "?"
        except Exception:
            size = "?"
    elif exists:
        size = f"{os.path.getsize(full_path) / 1024:.1f} KB"
    else:
        size = "—"
    legacy_rows.append({
        "path": path,
        "exists": exists,
        "size": size,
        "reason": reason,
    })

df_legacy = pd.DataFrame(legacy_rows)
print(f"Legacy/exploratory items: {len(df_legacy)} ({df_legacy['exists'].sum()} exist)\n")
display(df_legacy)

## 11. Persistent Metadata History

The `dataframes/persistent_metadata/` directory contains timestamped backups of the dataset
metadata, tracking how it evolved during data collection and processing.

In [ ]:
# Persistent metadata: timeline of backups
persist_dir = os.path.join(data_root, "dataframes", "persistent_metadata")
if os.path.isdir(persist_dir):
    persist_files = sorted(os.listdir(persist_dir))
    print(f"Persistent metadata files: {len(persist_files)}\n")

    # Parse dates from gold_dataset_df_backup_YYYYMMDD.csv files
    backup_dates = []
    for f in persist_files:
        m = re.search(r"backup_(\d{8})", f)
        if m:
            try:
                dt = pd.to_datetime(m.group(1), format="%Y%m%d")
                size_kb = os.path.getsize(os.path.join(persist_dir, f)) / 1024
                backup_dates.append({"file": f, "date": dt, "size_kb": size_kb})
            except ValueError:
                pass

    if backup_dates:
        df_backups = pd.DataFrame(backup_dates).sort_values("date")
        print(f"Dated backups: {len(df_backups)}")
        print(f"Date range: {df_backups['date'].min().date()} → {df_backups['date'].max().date()}\n")

        fig, ax = plt.subplots(figsize=(12, 4))
        ax.scatter(df_backups["date"], df_backups["size_kb"], s=50, edgecolor="black", zorder=3)
        ax.plot(df_backups["date"], df_backups["size_kb"], ls="--", alpha=0.5, color="gray")
        ax.set(title="Metadata Backup Timeline", xlabel="Date", ylabel="File size (KB)")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

    # Categorize all files
    categories = Counter()
    for f in persist_files:
        if "gold_dataset" in f:
            categories["gold_dataset backups"] += 1
        elif "smartflat_dataset" in f:
            categories["smartflat_dataset backups"] += 1
        elif "metadata" in f.lower():
            categories["metadata files"] += 1
        elif "dataset_df" in f:
            categories["host-specific datasets"] += 1
        else:
            categories["other"] += 1

    print("File categories:")
    for cat, count in categories.most_common():
        print(f"  {cat}: {count}")

## 12. Change-Point Detection Results

Load the deployed CPD parameters (lambda values) and visualize the segmentation
characteristics across participants.

See: `outputs/gold-change-point-detection-prototypes-deployment/lambda_optimal.csv`

In [ ]:
# Load lambda_optimal.csv — deployed CPD parameters
lambda_path = os.path.join(data_root, "outputs",
                           "gold-change-point-detection-prototypes-deployment",
                           "lambda_optimal.csv")
if os.path.isfile(lambda_path):
    df_lambda = pd.read_csv(lambda_path)
    print(f"CPD lambda results: {len(df_lambda)} rows, {df_lambda.shape[1]} columns")
    print(f"Columns: {list(df_lambda.columns)}\n")
    display(df_lambda.head(5))

    # Find numeric columns for visualization
    numeric_cols = df_lambda.select_dtypes(include=[np.number]).columns.tolist()
    # Look for lambda, n_changepoints, n_segments type columns
    lambda_cols = [c for c in numeric_cols if "lambda" in c.lower() or "penalty" in c.lower()]
    cpt_cols = [c for c in numeric_cols if "changepoint" in c.lower() or "cpt" in c.lower()
                or "segment" in c.lower() or "n_cpts" in c.lower()]

    plot_cols = lambda_cols + cpt_cols
    if not plot_cols:
        plot_cols = numeric_cols[:4]  # fallback: first 4 numeric columns

    if plot_cols:
        n_plots = min(len(plot_cols), 4)
        fig, axes = plt.subplots(1, n_plots, figsize=(5 * n_plots, 4))
        if n_plots == 1:
            axes = [axes]
        for ax, col in zip(axes, plot_cols[:n_plots]):
            vals = df_lambda[col].dropna()
            ax.hist(vals, bins=30, edgecolor="black", alpha=0.7)
            ax.axvline(vals.median(), color="red", ls="--",
                       label=f"Median={vals.median():.2f}")
            ax.set(title=col, xlabel="Value", ylabel="Count")
            ax.legend(fontsize=8)
        plt.suptitle("Change-Point Detection Parameters", y=1.02)
        plt.tight_layout()
        plt.show()
else:
    print(f"Lambda file not found: {lambda_path}")

## 13. Summary

Key findings from this data inventory:

- **198 participant folders** in `cuisine/` with standardized modality subfolders
- **Feature extraction** coverage varies by type — video representations and hand landmarks are the most complete
- **8 rounds** of recursive prototyping annotations across multiple kernel variants
- **Clinical data** covers diagnostic groups (Control/TBI/RIL) with neuropsychological assessments
- **141 + 25 UUID-based** CPD grid search experiments with deployed parameters
- **Legacy folders** identified for potential cleanup (see `DATA_INVENTORY.md` Appendix A)

For details on each folder's production status and pipeline origin, see **`DATA_INVENTORY.md`**.